# Model 16 — Bi-LSTM

This experiment uses a Bidirectional LSTM neural network.

The model:
- uses the same cleaned `combined_text` created by `preprocess_data()`;
- learns its own integer vocabulary using a Keras `Tokenizer`;
- pads every article to the same sequence length;
- reads each sequence forwards and backwards through a Bi-LSTM;
- predicts Fake (`0`) or Real (`1`).

In [ ]:
import os
import sys

os.chdir("/Users/karima/Ironhack-challenges/fake-news-nlp-classification")
sys.path.append(os.getcwd())

print(os.getcwd())

In [ ]:
# Run once if TensorFlow is not installed:
# %pip install tensorflow

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.model_selection import train_test_split

from tensorflow.keras.callbacks import EarlyStopping

from src.data_loader import load_data, prepare_features_and_labels, split_data
from src.preprocessing import preprocess_data
from src.bilstm import create_lstm_features, build_bilstm_model
from src.evaluator import save_confusion_matrix, save_metrics_plot
from src.experiment_tracker import save_experiment_result
from src.model_manager import save_vectorizer
from src.config import MODELS_DIR, PLOT_DIR

In [ ]:
# Class constants
MODEL_ID = "exp_16"
MODEL_NAME = "Bi-LSTM"
FEATURES = "Keras integer sequences with trainable embedding layer"
PREPROCESSING = "Lowercase + HTML removal + stopword removal + lemmatization"
ALGORITHM = "Bidirectional LSTM"
NOTES = "Bi-LSTM model using padded text sequences and a trainable embedding layer."

MODEL_PATH = MODELS_DIR / "bilstm_model.keras"
TOKENIZER_PATH = MODELS_DIR / "bilstm_tokenizer.pkl"

MAX_WORDS = 20000
MAX_LENGTH = 300
EMBEDDING_DIM = 100
EPOCHS = 10
BATCH_SIZE = 64

In [ ]:
# Load the dataset
data = load_data()

# Extract features X and labels y
X, y = prepare_features_and_labels(data)

# Keep the original 80/20 project split
X_train_full, X_test, y_train_full, y_test = split_data(X, y)

# Create a validation set from the training portion
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.15,
    random_state=42,
    stratify=y_train_full
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

In [ ]:
# Clean title and text and create combined_text
X_train_clean = preprocess_data(X_train)
X_val_clean = preprocess_data(X_val)
X_test_clean = preprocess_data(X_test)

In [ ]:
# Learn the tokenizer only from the training data
X_train_padded, X_test_padded, tokenizer = create_lstm_features(
    X_train_clean,
    X_test_clean,
    max_words=MAX_WORDS,
    max_length=MAX_LENGTH
)

# Validation data must use the same fitted tokenizer
val_sequences = tokenizer.texts_to_sequences(
    X_val_clean["combined_text"]
)

from tensorflow.keras.preprocessing.sequence import pad_sequences

X_val_padded = pad_sequences(
    val_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

print("Train shape:", X_train_padded.shape)
print("Validation shape:", X_val_padded.shape)
print("Test shape:", X_test_padded.shape)

In [ ]:
# Build the model
model = build_bilstm_model(
    max_words=MAX_WORDS,
    embedding_dim=EMBEDDING_DIM,
    max_length=MAX_LENGTH
)

model.summary()

In [ ]:
# Stop training when validation loss stops improving
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping]
)

In [ ]:
# Predict train and test probabilities
train_probabilities = model.predict(X_train_padded).ravel()
test_probabilities = model.predict(X_test_padded).ravel()

# Convert probabilities to class labels
train_predictions = (train_probabilities >= 0.5).astype(int)
predictions = (test_probabilities >= 0.5).astype(int)

train_accuracy = accuracy_score(y_train, train_predictions)

metrics = {
    "test_accuracy": accuracy_score(y_test, predictions),
    "precision": precision_score(y_test, predictions, zero_division=0),
    "recall": recall_score(y_test, predictions, zero_division=0),
    "f1_score": f1_score(y_test, predictions, zero_division=0)
}

metrics

In [ ]:
# Save and show metrics plot
metrics_plot_path = save_metrics_plot(
    metrics,
    "16_bilstm_metrics"
)

# Save and show confusion matrix
confusion_matrix_path = save_confusion_matrix(
    y_test,
    predictions,
    "16_bilstm_confusion_matrix"
)

In [ ]:
# Save training history plot
PLOT_DIR.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Bi-LSTM Training Accuracy")
plt.legend()

training_plot_path = PLOT_DIR / "16_bilstm_training_accuracy.png"
plt.savefig(training_plot_path, bbox_inches="tight")
plt.show()

In [ ]:
# Save model and fitted tokenizer
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

model.save(MODEL_PATH)

tokenizer_path = save_vectorizer(
    tokenizer,
    "bilstm_tokenizer.pkl"
)

print(f"Model saved to: {MODEL_PATH}")
print(f"Tokenizer saved to: {tokenizer_path}")

In [ ]:
# Save experiment result
experiment = {
    "model_id": MODEL_ID,
    "model_name": MODEL_NAME,
    "features": FEATURES,
    "preprocessing": PREPROCESSING,
    "algorithm": ALGORITHM,
    "train_accuracy": train_accuracy,
    "test_accuracy": metrics["test_accuracy"],
    "precision": metrics["precision"],
    "recall": metrics["recall"],
    "f1_score": metrics["f1_score"],
    "notes": NOTES,
    "model_path": MODEL_PATH,
}

tracking = save_experiment_result(**experiment)

tracking.tail()